In [ ]:
import pandas as pd
from pathlib import Path


intermediates_path = Path("intermediates")
gtdb_output_path = intermediates_path / "cwd" / "gtdbtk" / "gtdb_out"

mags_to_classify_path = intermediates_path / "mags_to_classify.csv"
bacteria_summary_path = gtdb_output_path / "gtdbtk.bac120.summary.tsv"
archaea_summary_path = gtdb_output_path / "gtdbtk.ar53.summary.tsv"

mags_metadata = pd.read_csv(mags_to_classify_path, low_memory=False)
bacteria_summary = pd.read_csv(bacteria_summary_path, delimiter="\t", low_memory=False)
archaea_summary = pd.read_csv(archaea_summary_path, delimiter="\t", low_memory=False)

In [ ]:
mags_classified_path = intermediates_path / "mags_classified.csv"


gtdb_summary = pd.concat([bacteria_summary, archaea_summary])[["user_genome", "classification"]]

# Taking only species level from string of such format:
# d__Bacteria;p__Pseudomonadota;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Escherichia;s__Escherichia coli
gtdb_summary["classification"] = gtdb_summary["classification"].apply(lambda x: x.split(";")[-1][3:])

# As filename is expected to be the same as spire_id, we just use it this way.
gtdb_single = gtdb_summary.rename(columns={"user_genome": "spire_id"})

mags_classified = gtdb_single.set_index("spire_id")\
    .join(mags_metadata.set_index("spire_id"), how="inner", validate="1:1")
mags_classified.to_csv(mags_classified_path)
mags_classified

In [ ]:
countries_count = mags_classified["geographic_location"].nunique()

countries_in = mags_classified.groupby("classification")["geographic_location"].nunique()
prevalent_species = countries_in[countries_in == countries_count].index
prevalent_mags = mags_classified[mags_classified["classification"].isin(prevalent_species)]
prevalent_mags

### Table 3

In [ ]:
prevalence_table = prevalent_mags.groupby(["geographic_location", "classification"]).size().unstack(0)
prevalence_table["s_prev"] = prevalence_table.prod(axis=1)
prevalence_table.index = prevalence_table.index.rename("Country")

prevalence_table = prevalence_table.sort_values(by="s_prev", ascending=False)

In [ ]:
best_by_mags = prevalence_table.head(5).index
best_by_mags

The mags generated here are passed to fastANI.

In [ ]:
mags_to_cluster_path = intermediates_path / "mags_to_cluster.csv"

mags_to_cluster = mags_classified[mags_classified["classification"].isin(best_by_mags)]
mags_to_cluster.to_csv(mags_to_cluster_path)

mags_to_cluster